# Result JSON

`dump_result_json` exports plain result data to strict JSON while preserving the distinction between an unbounded value, an undefined value and a missing value. `load_result_json` restores that data as Python dictionaries, lists and numbers. These functions export result records; `save_model` and `load_model` remain the interfaces for saving and loading fitted models.

The existing `to_dict()` methods preserve numerical infinities and NaN for compatibility with calculations on their returned records. Those values are excluded by the JSON standard. Pass the record to `dump_result_json`, or use `ConformalInterval.to_json()` for interval results.

## Nonfinite values

This example has a finite width of 2 response units, an unbounded width, an undefined standard error and an absent optional entry. The input is a constructed export record that shows the format.

In [ ]:
import json
import numpy as np
import mboostrs as mb

record = {"width": np.array([2.0, np.inf]), "standard_error": np.float64(np.nan), "optional": None}
document = mb.dump_result_json(record, indent=2)

# A standard strict consumer reads null placeholders and explicit category metadata.
envelope = json.loads(document)
print(document)

The `mboostrs.result-json/1` envelope has three fields. `schema` holds the string `mboostrs.result-json/1`. `data` contains ordinary JSON values, with null placeholders for nonfinite numbers. `nonfinite` maps each such location, written as a JSON Pointer (a path such as `/a/0`, meaning key `a` and then array element 0), to `positive_infinity`, `negative_infinity` or `nan`. A genuine null has no category entry. The entire user record sits inside `data`, so its keys cannot collide with the envelope's metadata.

## Restored records

The decoder restores numerical categories without creating a model or executing code from the record. NumPy arrays and tuples become Python lists.

In [ ]:
restored = mb.load_result_json(document)
assert restored["width"][0] == 2.0
assert np.isposinf(restored["width"][1])
assert np.isnan(restored["standard_error"])
assert restored["optional"] is None
print("Finite, unbounded, undefined and missing values retain their distinct meanings.")

## Interval exports

For an interval object `band`, `band.to_json(include_arrays=True)` includes row endpoints and predictions. It uses the same envelope as `mb.dump_result_json(band.to_dict(include_arrays=True))`. Decode either with `mb.load_result_json(document)`. A count interval can retain both an infinite upper endpoint and zero cardinality for an empty count set; neither becomes an unexplained missing value.

The strict encoder accepts JSON-compatible scalars, arrays, tuples and mappings with string keys. It refuses unsupported objects and ambiguous non-string mapping keys. The decoder refuses unknown schemas, duplicate keys, unknown nonfinite categories, non-null placeholders, unmatched paths and nonstandard `Infinity`/`NaN` tokens. Existing result dictionaries keep their shape and float values; opting into the strict format is explicit.